In [ ]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

In [ ]:


spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

In [ ]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

In [ ]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

In [ ]:
from delta.tables import DeltaTable

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'

    # Ler CSV com inferência de schema
    df = spark.read \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .csv(csv_path)

    # Gravar como Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)

    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversao concluida! {len(csv_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')

In [ ]:
print('Validando tabelas Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'

    # Verificar se é Delta
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)

    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

In [ ]:
response = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET, Delimiter='/')
tabelas = [p['Prefix'].rstrip('/') for p in response.get('CommonPrefixes', [])]

for tabela in tabelas:
    print(f'\n--- {tabela.upper()} ---')
    spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)

In [ ]:
spark.stop()
print('SparkSession finalizada.')